# TP3 - Sémaphores, files de messages et interruptions
## Synchronisation et partage de ressources avec FreeRTOS

**ECUE : Atelier Systèmes Temps Réel**  
**Support matériel : Arduino AVR + FreeRTOS**

Ce notebook reprend le TP3. Les cellules Python modélisent les sémaphores ; les implémentations matérielles se réalisent dans les exemples Arduino 012 à 016.

## Objectifs

- créer un sémaphore ;
- acquérir et libérer un sémaphore ;
- synchroniser une tâche avec une interruption ;
- utiliser un sémaphore à compte ;
- transférer des données entre une interruption et une tâche avec des queues ;
- protéger une ressource avec un mutex et une section critique.

FreeRTOS distingue le sémaphore binaire, le sémaphore à compte et le sémaphore d'exclusion mutuelle (mutex).

## API des sémaphores

```cpp
SemaphoreHandle_t binary = xSemaphoreCreateBinary();
SemaphoreHandle_t counting = xSemaphoreCreateCounting(10, 0);
SemaphoreHandle_t mutex = xSemaphoreCreateMutex();
xSemaphoreTake(binary, portMAX_DELAY);
xSemaphoreGive(binary);
xSemaphoreGiveFromISR(binary, &higherPriorityTaskWoken);
```

`xSemaphoreTake` bloque la tâche jusqu'à la disponibilité du sémaphore ou l'expiration de `xTicksToWait`. Une fonction d'interruption doit utiliser les variantes `FromISR`.

## Choisir la bonne primitive

| Primitive | Usage principal | Propriété importante |
|---|---|---|
| Sémaphore binaire | signaler un événement entre ISR et tâche | un seul jeton, pas de propriétaire |
| Sémaphore à compte | mémoriser plusieurs événements ou ressources identiques | valeur comprise entre 0 et `uxMaxCount` |
| Mutex | protéger une ressource partagée | propriétaire unique et héritage de priorité |

Un sémaphore binaire n'est pas un mutex : une tâche peut donner un sémaphore binaire sans être celle qui l'a acquis. Un mutex doit être libéré par sa tâche propriétaire.

## API complémentaires

```cpp
xSemaphoreTake(semaphore, xTicksToWait);
xSemaphoreGive(semaphore);
xSemaphoreGiveFromISR(semaphore, &higherPriorityTaskWoken);
uxSemaphoreGetCount(semaphore);
```

`uxSemaphoreGetCount()` permet d'observer le nombre de jetons d'un sémaphore à compte. Depuis une interruption, utiliser uniquement les fonctions `FromISR` et traiter `pxHigherPriorityTaskWoken` avant de quitter l'ISR.

Dans cette version AVR, vérifier dans `FreeRTOSConfig.h` que les options nécessaires sont activées, notamment `INCLUDE_vTaskSuspend`, `configUSE_MUTEXES` et `configUSE_COUNTING_SEMAPHORES`.

In [ ]:
class SemaphoreBinaire:
    """Modele d'un semaphore binaire : 0 ou 1 jeton."""

    def __init__(self, disponible=False):
        self.disponible = disponible

    def take(self):
        if not self.disponible:
            return False
        self.disponible = False
        return True

    def give(self):
        self.disponible = True
        return True


class SemaphoreCompteur:
    """Modele d'un semaphore a compte borne."""

    def __init__(self, maximum, initial=0):
        if maximum < 1 or not 0 <= initial <= maximum:
            raise ValueError('Compte initial ou maximum invalide.')
        self.maximum = maximum
        self.valeur = initial

    def take(self):
        if self.valeur == 0:
            return False
        self.valeur -= 1
        return True

    def give(self):
        if self.valeur == self.maximum:
            return False
        self.valeur += 1
        return True


class Mutex:
    """Modele d'un mutex avec un proprietaire unique."""

    def __init__(self):
        self.proprietaire = None

    def take(self, tache):
        if self.proprietaire is not None:
            return False
        self.proprietaire = tache
        return True

    def give(self, tache):
        if self.proprietaire != tache:
            return False
        self.proprietaire = None
        return True

binaire = SemaphoreBinaire()
compteur = SemaphoreCompteur(maximum=2, initial=0)
mutex = Mutex()
resultats = {
    'binaire': [binaire.take(), binaire.give(), binaire.take()],
    'compteur': [compteur.give(), compteur.give(), compteur.give(), compteur.take()],
    'mutex': [mutex.take('Task1'), mutex.take('Task2'), mutex.give('Task1')],
}
resultats

## Exercice 1 - Sémaphore binaire et interruption

Ouvrir `Example012`, observer `vHandlerTask` et `vPeriodicTask`, puis réaliser le montage avec bouton sur `INT0`. L'interruption libère le sémaphore avec `xSemaphoreGiveFromISR()`, ce qui débloque la tâche handler.

**À réaliser :**

1. réaliser le montage ;
2. modifier l'exemple pour tenir compte du bouton et observer le changement de contexte.

Référence : [`Example012.ino`](FreeRTOS_AVR/examples/FreeRTOSBook/Example012/Example012.ino).

## Exercice 2 - Sémaphore à compte

Ouvrir `Example013`. La création utilise :

```cpp
xSemaphoreCreateCounting(uxMaxCount, uxInitialCount);
```

Modifier la valeur maximale et le nombre d'appels à `xSemaphoreGiveFromISR()` dans l'interruption. Observer les événements mémorisés lorsque le handler est temporairement bloqué.

Référence : [`Example013.ino`](FreeRTOS_AVR/examples/FreeRTOSBook/Example013/Example013.ino).

In [ ]:
def semaphore_compteur(maximum, initial, liberations, acquisitions):
    valeur = initial
    journal = []
    for _ in range(liberations):
        valeur = min(maximum, valeur + 1)
        journal.append(('give', valeur))
    for _ in range(acquisitions):
        obtenu = valeur > 0
        if obtenu:
            valeur -= 1
        journal.append(('take', obtenu, valeur))
    return journal

semaphore_compteur(3, 0, 5, 5)

## Exercice 3 - Files de messages et interruptions

Ouvrir `Example014` et étudier `xQueueReceiveFromISR()` et `xQueueSendToBackFromISR()`. Modifier la taille des deux files et expliquer l'effet sur `vIntegerGenerator` et `vStringPrinter`.

```cpp
xQueueSendToBackFromISR(queue, &value, &higherPriorityTaskWoken);
xQueueReceiveFromISR(queue, &buffer, &higherPriorityTaskWoken);
```

Référence : [`Example014.ino`](FreeRTOS_AVR/examples/FreeRTOSBook/Example014/Example014.ino).

## Exercice 4 - Mutex et section critique

Ouvrir `Example015`, observer les tâches `Print1` et `Print2`, puis expliquer comment `xSemaphoreCreateMutex()` empêche deux tâches d'écrire simultanément dans une ressource partagée. Le mutex FreeRTOS applique l'héritage de priorité.

```cpp
SemaphoreHandle_t mutex = xSemaphoreCreateMutex();
xSemaphoreTake(mutex, portMAX_DELAY);
// accès exclusif à la ressource
xSemaphoreGive(mutex);
```

Référence : [`Example015.ino`](FreeRTOS_AVR/examples/FreeRTOSBook/Example015/Example015.ino).

## Inversion de priorité et héritage

Scénario à étudier dans `Example015` :

1. `Low` acquiert le mutex et entre dans la section critique ;
2. `High` devient prête et demande le même mutex ;
3. `High` se bloque ;
4. `Medium`, qui n'utilise pas le mutex, devient prête.

Sans héritage de priorité, `Medium` peut retarder `Low`, ce qui retarde indirectement `High`. Avec un mutex FreeRTOS, la priorité de `Low` peut être temporairement élevée pour lui permettre de libérer rapidement la ressource.

**Mesure demandée :** comparer le temps d'attente de `High` avec un mutex et avec une protection mal choisie. Ne jamais garder un mutex pendant une impression série, un délai ou une opération longue.

In [ ]:
def scenario_inversion(heritage=False):
    """Compare un ordonnancement sans et avec heritage de priorite."""
    priorites = {'Low': 1, 'Medium': 2, 'High': 3}
    journal = []
    mutex_libre = False

    for tick in range(6):
        if tick == 0:
            mutex_libre = False
            journal.append((tick, 'Low', 'acquiert le mutex'))
            continue
        if tick == 1:
            journal.append((tick, 'High', 'se bloque sur le mutex'))
            if heritage:
                priorites['Low'] = priorites['High']
        if tick in (2, 3):
            tache = max(priorites, key=priorites.get)
            journal.append((tick, tache, 'execute'))
        if tick == 4:
            mutex_libre = True
            priorites['Low'] = 1
            journal.append((tick, 'Low', 'libere le mutex'))
        if tick == 5 and mutex_libre:
            journal.append((tick, 'High', 'reprend le mutex'))

    return journal

sans_heritage = scenario_inversion(False)
avec_heritage = scenario_inversion(True)
sans_heritage, avec_heritage

## Exercice 5 - Gestionnaire de ressource

Ouvrir `Example016` et observer les tâches `Print1`, `Print2` et `Gatekeeper`. Le gatekeeper est la seule tâche autorisée à accéder à la ressource ; les autres tâches lui transmettent leurs demandes avec `xQueueSendToFrontFromISR()` ou une queue dédiée.

**Compte rendu :** décrire la différence entre synchroniser un événement, compter des événements, protéger une ressource et déléguer l'accès à une tâche unique.

Référence : [`Example016.ino`](FreeRTOS_AVR/examples/FreeRTOSBook/Example016/Example016.ino).

In [ ]:
def section_critique(taches, tours=3):
    journal = []
    for tour in range(tours):
        for tache in taches:
            journal.append((tour, tache, 'acquisition'))
            journal.append((tour, tache, 'ressource protégée'))
            journal.append((tour, tache, 'libération'))
    return journal

section_critique(['Print1', 'Print2'], 2)

## Règles ISR, validation et compte rendu

Une ISR FreeRTOS doit rester courte :

- utiliser uniquement les API `FromISR` ;
- ne pas appeler `Serial.print`, `vTaskDelay` ou `xSemaphoreTake` depuis l'ISR ;
- initialiser `pxHigherPriorityTaskWoken` à `pdFALSE` ;
- demander un changement de contexte si une tâche prioritaire a été débloquée ;
- transférer le traitement long à une tâche réveillée par sémaphore ou queue.

### Checklist de validation

- [ ] Le sémaphore est créé avant le démarrage du scheduler.
- [ ] Le sémaphore binaire est pris initialement si l'événement doit venir de l'interruption.
- [ ] Les limites `uxMaxCount` et `uxInitialCount` sont testées.
- [ ] Les accès au mutex sont encadrés par `take` puis `give` dans la même tâche.
- [ ] La section critique ne contient ni délai ni impression série longue.
- [ ] Les queues ISR utilisent `pxHigherPriorityTaskWoken`.
- [ ] Le câblage et le front d'interruption sont documentés.
- [ ] Les observations série sont comparées aux états attendus.

Pour le compte rendu, joindre le schéma de câblage, les observations série, les valeurs de compte testées, une analyse des priorités et une comparaison entre sémaphore binaire, sémaphore à compte et mutex.